# 08 FastAPI Gateway

This notebook explains and starts the adapter routing gateway.

```mermaid
flowchart LR
    A[Client request] --> B[FastAPI gateway]
    B --> C{Adapter}
    C -->|finance| D[vLLM model finance]
    C -->|legal| E[vLLM model legal]
    C -->|healthcare| F[vLLM model healthcare]
    C -->|fallback| G[vLLM model base]
```

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PROJECT_SPEC.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from llmops_demo.settings import settings, ensure_dirs

cfg = settings()
print(f"Project root: {PROJECT_ROOT}")
print(f"Base model: {cfg.base_model}")
print(f"Adapters: {cfg.adapters}")

## Routing rules

The gateway accepts an explicit `adapter` or `domain`. If none is provided, it uses simple domain keywords.

In [ ]:
from serving.gateway import infer_adapter, ChatMessage

examples = [
    "Explain revenue concentration risk.",
    "Summarize this limitation of liability clause.",
    "What does prior authorization mean?",
    "Tell me a neutral productivity tip.",
]
for prompt in examples:
    adapter = infer_adapter([ChatMessage(role="user", content=prompt)])
    print(f"{adapter}: {prompt}")

## Start gateway

Run this in a terminal. Expected output: Uvicorn serving on `http://localhost:8080`.

In [ ]:
print("make api")

## Test gateway request

Expected output: an OpenAI-compatible chat completion plus `routed_adapter`.

In [ ]:
import requests

payload = {
    "adapter": "finance",
    "messages": [{"role": "user", "content": "Explain revenue concentration risk."}],
    "temperature": 0.2,
    "max_tokens": 160,
}
response = requests.post(f"{cfg.api_base_url}/chat", json=payload, timeout=60)
print(response.status_code)
print(response.text[:1200])